# lasso

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, LassoCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Directory for saving plots
plot_dir = "/Users/alberto/Documents/projects/GWP_1/MLFinance/MLFinance/gwp1-2"
os.makedirs(plot_dir, exist_ok=True)

# Set random seed for reproducibility
np.random.seed(42)

# Parameters for synthetic high-dimensional financial data
n_samples = 200      # number of observations (e.g., stocks or time periods)
n_features = 100     # number of predictors (factors, indicators)
n_informative = 10   # true number of non-zero coefficients
noise_level = 1.0

logging.info(f"Generating synthetic data: {n_samples} samples, {n_features} features, {n_informative} informative")

# Generate correlated design matrix X
X = np.random.randn(n_samples, n_features)
# Introduce correlation
corr = 0.5
for i in range(n_features):
    for j in range(i+1, n_features):
        X[:, j] += corr * X[:, i]

# True sparse coefficients (only first 10 are non-zero)
beta_true = np.zeros(n_features)
beta_true[:n_informative] = np.random.uniform(-2, 2, n_informative)  # random signs and magnitudes

# Generate target y = X @ beta_true + noise
y = X @ beta_true + np.random.normal(0, noise_level, n_samples)

logging.info(f"True non-zero coefficients: {np.sum(beta_true != 0)}")
logging.info(f"Range of true coefficients: [{beta_true.min():.2f}, {beta_true.max():.2f}]")

# Standardise features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

logging.info(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

# Fit Lasso with fixed alpha=0.1
lasso = Lasso(alpha=0.1, max_iter=10000, random_state=42)
lasso.fit(X_train, y_train)

# Predictions and performance
y_pred = lasso.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
n_selected = np.sum(lasso.coef_ != 0)

logging.info(f"LASSO (alpha=0.1) selected {n_selected} features out of {n_features}")
logging.info(f"Test MSE: {mse:.4f}")

# Plot 1: Coefficient paths (varying alpha)
alphas = np.logspace(-2, 1, 50)
coefs = []
for a in alphas:
    model = Lasso(alpha=a, max_iter=10000)
    model.fit(X_train, y_train)
    coefs.append(model.coef_)

plt.figure(figsize=(10, 6))
plt.plot(alphas, coefs)
plt.xscale('log')
plt.xlabel('Alpha (regularisation strength)')
plt.ylabel('Coefficients')
plt.title('LASSO Coefficient Paths')
plt.axvline(0.1, color='red', linestyle='--', label='Selected alpha=0.1')
plt.legend()
plt.grid(True)
path_plot = os.path.join(plot_dir, "lasso_coefficient_paths.png")
plt.savefig(path_plot)
plt.close()
logging.info(f"Coefficient paths plot saved to {path_plot}")

# Plot 2: True vs Estimated coefficients
plt.figure(figsize=(10, 6))
plt.scatter(range(n_features), beta_true, color='blue', label='True coefficients', s=50)
plt.scatter(range(n_features), lasso.coef_, color='red', label='LASSO estimated (alpha=0.1)', alpha=0.7)
plt.axhline(0, color='black', linewidth=0.5)
plt.xlabel('Feature index')
plt.ylabel('Coefficient value')
plt.title('True vs LASSO Estimated Coefficients')
plt.legend()
plt.grid(True)
true_est_plot = os.path.join(plot_dir, "lasso_true_vs_estimated.png")
plt.savefig(true_est_plot)
plt.close()
logging.info(f"True vs estimated plot saved to {true_est_plot}")

# Plot 3: Predictions vs Actual on test set
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.7)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
plt.xlabel('Actual returns')
plt.ylabel('Predicted returns')
plt.title('LASSO Predictions vs Actual (Test Set)')
plt.grid(True)
pred_plot = os.path.join(plot_dir, "lasso_predictions_vs_actual.png")
plt.savefig(pred_plot)
plt.close()
logging.info(f"Predictions vs actual plot saved to {pred_plot}")

# Optional: LassoCV to show automatic alpha selection
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso_cv.fit(X_train, y_train)
logging.info(f"LassoCV selected alpha: {lasso_cv.alpha_:.4f}")
logging.info(f"LassoCV selected {np.sum(lasso_cv.coef_ != 0)} features")

logging.info("LASSO illustration completed successfully.")

2025-12-13 09:29:12,889 - INFO - Generating synthetic data: 200 samples, 100 features, 10 informative
2025-12-13 09:29:12,897 - INFO - True non-zero coefficients: 10
2025-12-13 09:29:12,898 - INFO - Range of true coefficients: [-1.88, 1.94]
2025-12-13 09:29:12,903 - INFO - Train shape: (140, 100), Test shape: (60, 100)
2025-12-13 09:29:12,911 - INFO - LASSO (alpha=0.1) selected 5 features out of 100
2025-12-13 09:29:12,911 - INFO - Test MSE: 3.6683
2025-12-13 09:29:13,144 - INFO - Coefficient paths plot saved to /Users/alberto/Documents/projects/GWP_1/MLFinance/MLFinance/gwp1-2/lasso_coefficient_paths.png
2025-12-13 09:29:13,200 - INFO - True vs estimated plot saved to /Users/alberto/Documents/projects/GWP_1/MLFinance/MLFinance/gwp1-2/lasso_true_vs_estimated.png
2025-12-13 09:29:13,236 - INFO - Predictions vs actual plot saved to /Users/alberto/Documents/projects/GWP_1/MLFinance/MLFinance/gwp1-2/lasso_predictions_vs_actual.png
2025-12-13 09:29:13,291 - INFO - LassoCV selected alpha: 0.

#  k-means

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Directory for saving plots
plot_dir = "/Users/alberto/Documents/projects/GWP_1/MLFinance/MLFinance/gwp1-2"
os.makedirs(plot_dir, exist_ok=True)

# Set random seed for reproducibility
np.random.seed(42)

# Parameters
n_stocks = 200      # number of assets
n_days = 252        # trading days in a year
n_regimes = 3       # true number of market regimes

# Simulate 3 regimes with different means and covariances
regime_means = [0.0008, -0.0005, 0.0000]   # daily returns: bull, bear, sideways
regime_stds = [0.015, 0.025, 0.010]
regime_sizes = [80, 60, 60]  # number of stocks per regime

# Generate returns data
returns = []
labels_true = []
for i in range(n_regimes):
    size = regime_sizes[i]
    mean = regime_means[i]
    std = regime_stds[i]
    # Correlated returns within regime
    cov = np.full((size, size), 0.3) * (std**2)
    np.fill_diagonal(cov, std**2)
    regime_returns = np.random.multivariate_normal(np.full(size, mean), cov, n_days)
    returns.append(regime_returns)
    labels_true.extend([i] * size)

# Concatenate into (n_days x n_stocks) matrix, then transpose to (n_stocks x n_days)
returns_matrix = np.hstack(returns).T  # shape: (n_stocks, n_days)

# Convert to DataFrame for clarity
df_returns = pd.DataFrame(returns_matrix, columns=[f"Day_{t}" for t in range(n_days)])

logging.info(f"Generated synthetic returns data: shape {df_returns.shape}")
logging.info(f"Regime distribution: {np.bincount(labels_true)} stocks per regime")

# Standardise features (stocks as rows, days as features)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_returns)

logging.info("Data standardised (zero mean, unit variance per feature)")

# Fit k-means with k=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
cluster_labels = kmeans.fit_predict(X_scaled)

logging.info(f"k-means converged in {kmeans.n_iter_} iterations")
logging.info(f"Final inertia (within-cluster SS): {kmeans.inertia_:.2f}")

# Count stocks per cluster
cluster_counts = np.bincount(cluster_labels)
logging.info(f"Cluster sizes: {cluster_counts}")

# Elbow method: inertia for k=1 to 10
inertias = []
K_range = range(1, 11)
for k in K_range:
    km_temp = KMeans(n_clusters=k, random_state=42, n_init='auto')
    km_temp.fit(X_scaled)
    inertias.append(km_temp.inertia_)
    print(f"k={k}, inertia={km_temp.inertia_:.2f}")

# Plot 1: Elbow curve
plt.figure(figsize=(8, 6))
plt.plot(K_range, inertias, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.grid(True)
elbow_path = os.path.join(plot_dir, "kmeans_elbow_plot.png")
plt.savefig(elbow_path)
plt.close()
logging.info(f"Elbow plot saved to {elbow_path}")

# Plot 2: 2D visualisation using PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='viridis', alpha=0.7)
plt.title('k-means Clustering Visualisation (PCA-reduced to 2D)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.colorbar(scatter, label='Cluster')
cluster_scatter_path = os.path.join(plot_dir, "kmeans_cluster_scatter.png")
plt.savefig(cluster_scatter_path)
plt.close()
logging.info(f"Cluster scatter plot saved to {cluster_scatter_path}")

# Plot 3: True regimes vs predicted clusters (for reference)
plt.figure(figsize=(10, 8))
scatter_true = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_true, cmap='plasma', alpha=0.7)
plt.title('True Underlying Regimes (for comparison)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
plt.colorbar(scatter_true, label='True Regime')
true_scatter_path = os.path.join(plot_dir, "kmeans_true_regimes.png")
plt.savefig(true_scatter_path)
plt.close()
logging.info(f"True regimes plot saved to {true_scatter_path}")

logging.info("k-means illustration completed successfully.")

2025-12-13 09:23:47,862 - INFO - Generated synthetic returns data: shape (200, 252)
2025-12-13 09:23:47,863 - INFO - Regime distribution: [80 60 60] stocks per regime
2025-12-13 09:23:47,866 - INFO - Data standardised (zero mean, unit variance per feature)
2025-12-13 09:23:47,949 - INFO - k-means converged in 3 iterations
2025-12-13 09:23:47,950 - INFO - Final inertia (within-cluster SS): 45577.03
2025-12-13 09:23:47,950 - INFO - Cluster sizes: [ 57 120  23]
2025-12-13 09:23:48,054 - INFO - Elbow plot saved to /Users/alberto/Documents/projects/GWP_1/MLFinance/MLFinance/gwp1-2/kmeans_elbow_plot.png
2025-12-13 09:23:48,139 - INFO - Cluster scatter plot saved to /Users/alberto/Documents/projects/GWP_1/MLFinance/MLFinance/gwp1-2/kmeans_cluster_scatter.png


k=1, inertia=50400.00
k=2, inertia=45811.82
k=3, inertia=45577.03
k=4, inertia=45089.61
k=5, inertia=40173.80
k=6, inertia=40004.30
k=7, inertia=39530.09
k=8, inertia=39121.32
k=9, inertia=38727.40
k=10, inertia=38328.53


2025-12-13 09:23:48,210 - INFO - True regimes plot saved to /Users/alberto/Documents/projects/GWP_1/MLFinance/MLFinance/gwp1-2/kmeans_true_regimes.png
2025-12-13 09:23:48,210 - INFO - k-means illustration completed successfully.


# PCA

In [5]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Directory for saving plots
plot_dir = "/Users/alberto/Documents/projects/GWP_1/MLFinance/MLFinance/gwp1-2"
os.makedirs(plot_dir, exist_ok=True)

# Set random seed for reproducibility
np.random.seed(42)

# Parameters: simulate returns for 500 stocks over 252 trading days
n_assets = 500
n_days = 252

logging.info(f"Generating synthetic asset returns: {n_assets} assets, {n_days} days")

# Simulate returns with 5 underlying factors + idiosyncratic noise
n_factors = 5

# Factor loadings: different exposure strengths (decreasing importance)
factor_loadings = np.random.randn(n_assets, n_factors)
loading_scales = np.array([1.5, 1.2, 1.0, 0.8, 0.6])
factor_loadings = factor_loadings * loading_scales  # shape: (n_assets, n_factors)

# Factor returns: daily returns with decreasing volatility
factor_vols = np.array([0.001, 0.0008, 0.0006, 0.0005, 0.0004])
factor_returns = np.random.randn(n_days, n_factors) * factor_vols  # proper broadcasting

# Common component: loadings @ factor_returns.T
common_component = factor_loadings @ factor_returns.T  # shape: (n_assets, n_days)

# Idiosyncratic noise
idiosyncratic = np.random.randn(n_assets, n_days) * 0.02

# Total returns
returns = common_component + idiosyncratic

# For PCA in finance, typically assets as rows, time periods as columns
X = returns  # shape: (n_assets, n_days)

logging.info(f"Returns matrix shape for PCA: {X.shape} (assets x days)")

# Standardise features (days)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit PCA
pca = PCA(random_state=42)
pca.fit(X_scaled)

# Explained variance
explained_variance_ratio = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance_ratio)

logging.info(f"First 5 components explain {cumulative_variance[4]:.1%} of variance")
logging.info(f"First 10 components explain {cumulative_variance[9]:.1%} of variance")
logging.info(f"Components needed for 90% variance: {np.argmax(cumulative_variance >= 0.90) + 1}")

# Plot 1: Scree plot
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(pca.explained_variance_) + 1), pca.explained_variance_, marker='o', label='Eigenvalues')
plt.title('Scree Plot')
plt.xlabel('Principal Component')
plt.ylabel('Eigenvalue (Variance)')
plt.grid(True)
plt.axhline(y=1, color='r', linestyle='--', label='Kaiser criterion (eigenvalue > 1)')
plt.legend()
scree_path = os.path.join(plot_dir, "pca_scree_plot.png")
plt.savefig(scree_path)
plt.close()
logging.info(f"Scree plot saved to {scree_path}")

# Plot 2: Cumulative explained variance
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', color='green')
plt.title('Cumulative Explained Variance Ratio')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Variance Explained')
plt.grid(True)
plt.axhline(y=0.90, color='r', linestyle='--', label='90% threshold')
plt.legend()
cumvar_path = os.path.join(plot_dir, "pca_cumulative_variance.png")
plt.savefig(cumvar_path)
plt.close()
logging.info(f"Cumulative variance plot saved to {cumvar_path}")

# Plot 3: Asset projection onto first two PCs
X_pca = pca.transform(X_scaled)
plt.figure(figsize=(10, 8))
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.6, s=10)
plt.title('Asset Projection onto First Two Principal Components')
plt.xlabel(f'PC1 ({explained_variance_ratio[0]:.1%} variance)')
plt.ylabel(f'PC2 ({explained_variance_ratio[1]:.1%} variance)')
plt.grid(True)
projection_path = os.path.join(plot_dir, "pca_asset_projection.png")
plt.savefig(projection_path)
plt.close()
logging.info(f"Asset projection plot saved to {projection_path}")

# Plot 4: Loadings heatmap (first 3 PCs, first 20 assets)
plt.figure(figsize=(12, 8))
loadings = pca.components_[:3, :20].T  # (20 assets, 3 PCs)
im = plt.imshow(loadings, cmap='coolwarm', aspect='auto')
plt.title('PCA Loadings Heatmap (First 3 PCs, First 20 Assets)')
plt.xlabel('Principal Component')
plt.ylabel('Asset Index')
plt.colorbar(im, label='Loading')
loadings_path = os.path.join(plot_dir, "pca_loadings_heatmap.png")
plt.savefig(loadings_path)
plt.close()
logging.info(f"Loadings heatmap saved to {loadings_path}")

logging.info("PCA illustration completed successfully.")

2025-12-13 09:37:01,294 - INFO - Generating synthetic asset returns: 500 assets, 252 days
2025-12-13 09:37:01,311 - INFO - Returns matrix shape for PCA: (500, 252) (assets x days)
2025-12-13 09:37:01,391 - INFO - First 5 components explain 5.8% of variance
2025-12-13 09:37:01,391 - INFO - First 10 components explain 10.9% of variance
2025-12-13 09:37:01,392 - INFO - Components needed for 90% variance: 167
2025-12-13 09:37:01,456 - INFO - Scree plot saved to /Users/alberto/Documents/projects/GWP_1/MLFinance/MLFinance/gwp1-2/pca_scree_plot.png
2025-12-13 09:37:01,495 - INFO - Cumulative variance plot saved to /Users/alberto/Documents/projects/GWP_1/MLFinance/MLFinance/gwp1-2/pca_cumulative_variance.png
2025-12-13 09:37:01,553 - INFO - Asset projection plot saved to /Users/alberto/Documents/projects/GWP_1/MLFinance/MLFinance/gwp1-2/pca_asset_projection.png
2025-12-13 09:37:01,632 - INFO - Loadings heatmap saved to /Users/alberto/Documents/projects/GWP_1/MLFinance/MLFinance/gwp1-2/pca_load